# 🌿 Leva-TTS — Quick Start

Generate your first Levantine-Arabic / English code-switching audio in a few cells.

**What this notebook does**
1. Installs `leva-tts`
2. Loads the model (auto-downloads the fine-tuned checkpoint from HuggingFace)
3. Synthesizes speech with a built-in speaker, a code-switch sentence, zero-shot cloning, and streaming

👉 Set **Runtime ▸ Change runtime type ▸ T4 GPU** first.

## 1. Check GPU

In [ ]:
# Verify a GPU runtime is active (Runtime ▸ Change runtime type ▸ T4 GPU)
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout or "⚠️ No GPU — switch the runtime to a GPU!")

## 2. Install Leva-TTS

In [ ]:
# Install Leva-TTS. Colab ships a compatible torch; we add the package + audio libs.
# (~2–3 min). Coqui TTS pulls a few deps; the pins below avoid version conflicts.
!pip install -q leva-tts
!apt-get -qq install -y espeak-ng > /dev/null
print("✅ Installed. If pip shows a dependency-resolver warning you can ignore it.")

> ⚠️ **If you see a "You must restart the runtime" message after install**, do
> **Runtime ▸ Restart session**, then continue from the next cell (do *not* re-run
> the install cell).

## 3. Load the model

First run downloads the checkpoint + 10 reference speakers (~2 GB) — give it a minute.

In [ ]:
from leva_tts import LevaTTS, SPEAKERS

tts = LevaTTS(device="cuda", preprocess_text=True, verbose=False)
print("Available speakers:", SPEAKERS)

## 4. Synthesize with a built-in speaker

`synthesize(text, speaker, **gen_params)` → `(wav, sr)`. `speaker` must be one of `SPEAKERS`.

In [ ]:
from IPython.display import Audio, display
import soundfile as sf

wav, sr = tts.synthesize(
    "هَلَّق أنا عم أشتغل على the new project اللي حكيتلك عنه.",
    speaker="Badr",
    temperature=0.65,
)
sf.write("badr.wav", wav, sr)
print(f"{len(wav)/sr:.2f}s @ {sr} Hz")
display(Audio(wav, rate=sr))

## 5. Try other speakers & a pure-Levantine line

In [ ]:
wav, sr = tts.synthesize("كيفك اليوم؟ إنت شو عم تعمل هَلَّق؟", speaker="Amina")
display(Audio(wav, rate=sr))

## 6. Zero-shot voice cloning

Upload your own 3–10 s reference clip and clone the voice. Run the cell, pick a `.wav`/`.mp3`.

In [ ]:
from google.colab import files
up = files.upload()                 # choose a short reference clip
ref_path = next(iter(up))

wav, sr = tts.zero_shot_synthesize("مرحبا، هَلَّق عم نجرب الـ voice cloning.", ref_path)
display(Audio(wav, rate=sr))

## 7. Streaming

`stream(...)` yields audio chunks as they're generated (useful for low-latency playback).

In [ ]:
import numpy as np
chunks = []
for ch in tts.stream("بِدِّي أحكيلك عن the new feature هَلَّق، رح تعجبك كتير.", speaker="Mona"):
    chunks.append(ch)
print(f"received {len(chunks)} chunks")
display(Audio(np.concatenate(chunks), rate=24000))

## 🎉 Done!

**Generation params** (optional on every method): `temperature`, `length_penalty`,
`repetition_penalty`, `top_k`, `top_p`, `speed`.

Next: [inference server](02_inference_server.ipynb) ·
[evaluation](03_evaluation.ipynb) · [Gradio app](04_gradio_app.ipynb).